# WattWise Grocery Store Energy Analysis

Run cells from top to bottom with the working directory set to the repository root or its data/ folder. Paths below resolve from either location.


## Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor
import joblib
from pathlib import Path
import numpy as np

# Resolve paths from either the repository root or data/ notebook directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "predictor.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "predictor.py").is_file():
    raise RuntimeError("Start the notebook from the repository root or data/ directory")
MODEL_PATH = PROJECT_ROOT / "models" / "random_forest.joblib"


## 1. Load the dataset

In [ ]:
df_electricity = pd.read_csv(
    PROJECT_ROOT / "data" / "adrenalin.1h.L07.B01.EM01.P.csv"
)

## 2. Inspect the raw data

In [ ]:
print(df_electricity.head())
print(df_electricity.columns)
print(df_electricity.shape)

df_electricity.info()

## 3. Convert timestamp to datetime

In [ ]:
df_electricity["timestamp [UTC]"] = pd.to_datetime(
    df_electricity["timestamp [UTC]"]
)

## 4. Rename columns

In [ ]:
df_electricity = df_electricity.rename(columns={
    "timestamp [UTC]": "timestamp",
    "value P [Total active power (kW)]": "power_kw"
})

## 5. Check data quality

In [ ]:
print("Start:", df_electricity["timestamp"].min())
print("End:", df_electricity["timestamp"].max())

print("Zero values:", (df_electricity["power_kw"] == 0).sum())
print("Negative values:", (df_electricity["power_kw"] < 0).sum())

print("Minimum power:", df_electricity["power_kw"].min())
print("Maximum power:", df_electricity["power_kw"].max())

## 6. Check hourly continuity

In [ ]:
time_diff = df_electricity["timestamp"].diff()

print(time_diff.value_counts())

## 7. Convert UTC to local Norway time

In [ ]:
df_electricity["local_timestamp"] = (
    df_electricity["timestamp"]
    .dt.tz_convert("Europe/Oslo")
)

## 8. Create time features

In [ ]:
df_electricity["hour"] = (
    df_electricity["local_timestamp"].dt.hour
)

df_electricity["day_of_week"] = (
    df_electricity["local_timestamp"].dt.day_name()
)

df_electricity["weekend"] = (
    df_electricity["day_of_week"]
    .isin(["Saturday", "Sunday"])
)

## 9. Analyze average power by hour

In [ ]:
average_by_hour = (
    df_electricity
    .groupby("hour")["power_kw"]
    .mean()
)

print("Peak hour:", average_by_hour.idxmax())
print("Peak average power:", average_by_hour.max())

print("Lowest hour:", average_by_hour.idxmin())
print("Lowest average power:", average_by_hour.min())

## 10. Compare weekday vs weekend

In [ ]:
average_by_weekend = (
    df_electricity
    .groupby("weekend")["power_kw"]
    .mean()
)

print(average_by_weekend)

## 11. Look at first 7 full days

In [ ]:
first_seven_days = df_electricity[
    (df_electricity["timestamp"] >= "2021-11-29 06:00:00+00:00") &
    (df_electricity["timestamp"] < "2021-12-06 06:00:00+00:00")
]

print(first_seven_days.shape)

## 12. Plot first 7 days

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    first_seven_days["local_timestamp"],
    first_seven_days["power_kw"]
)

plt.title("Grocery Store Power Demand - First 7 Days")
plt.xlabel("Local Time")
plt.ylabel("Power (kW)")

plt.show()

## 13. Average Power by Hour: Weekday vs Weekend

In [ ]:
average_by_hour_weekend = (
    df_electricity
    .groupby(["hour","weekend"]).power_kw.mean()
)
print(average_by_hour_weekend)

average_by_hour_weekend = average_by_hour_weekend.unstack()


x = average_by_hour_weekend.index
weekend = average_by_hour_weekend[True]
weekday = average_by_hour_weekend[False]

plt.plot(x, weekend, label="Weekend")
plt.plot(x, weekday, label="Weekday")
plt.xlabel("Hour of Day")
plt.ylabel("Average Power (kW)")
plt.title("Average Grocery Store Power Demand by Hour: Weekday vs Weekend")
plt.legend()
plt.show()

The graph shows that average power demand is lower on weekends than on weekdays, 
especially from around 5 AM through the evening. Overnight, weekday and weekend power demand are much more similar.

## 14. Check Pattern Differences Between Months

In [ ]:
df_electricity["month"] = df_electricity["local_timestamp"].dt.month

print(df_electricity.month)


average_by_month = df_electricity.groupby("month").power_kw.mean()

print(average_by_month)

## 15. Prepare Features and Target

In [ ]:
X = df_electricity[["hour","weekend","month"]]
y = df_electricity["power_kw"]

print(X.shape)

print(X.head())
print(y.head())

## 16. Split Training and Test Data

In [ ]:
split_index = int(len(df_electricity) * 0.8)

print(split_index)
print(len(df_electricity))

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(X_train.shape)

print(X_test.shape)

print(y_train.shape)

print(y_test.shape)

## 17. Train Linear Regression

In [ ]:
model = LinearRegression()

model.fit(X=X_train,y=y_train)

## 18. Generate Predictions

In [ ]:
y_pred = model.predict(X_test)

print(y_pred[:5])
print(y_test.head())

Finding the Mean Absolute Error to establish a baseline

In [ ]:
mae = mean_absolute_error(y_test,y_pred)

print(mae)

In [ ]:
print(y_test.mean())

Baseline Linear Regression
Features: hour, weekend, month
Test MAE: 11.59 kW
Average test demand: 48.15 kW

In [ ]:
print(r2_score(y_test,y_pred))

Baseline model results:
Linear Regression using hour, weekend, and month produced an MAE of 11.59 kW and an R² of 0.085. This indicates that the model captures only a small portion of the variation in grocery-store power demand.

In [ ]:
rf_model = RandomForestRegressor(random_state=42)

In [ ]:
rf_model.fit(X=X_train, y=y_train)

rf_pred = rf_model.predict(X_test)
print(rf_pred[:5])
print(y_test.head())

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print(rf_mae)
print(rf_r2)

In [ ]:
test_timestamps = df_electricity["local_timestamp"].iloc[split_index:]

plt.figure(figsize=(12, 5))

plt.plot(
    test_timestamps.iloc[:168],
    y_test.iloc[:168],
    label="Actual Power"
)

plt.plot(
    test_timestamps.iloc[:168],
    rf_pred[:168],
    label="Predicted Power"
)

plt.xlabel("Time")
plt.ylabel("Power (kW)")
plt.title("Random Forest: Actual vs Predicted Power - First Test Week")
plt.legend()
plt.show()

In [ ]:
rf_error = abs(y_test - rf_pred)

print(rf_error.head())

In [ ]:
largest_errors = rf_error.sort_values(ascending=False)
print(largest_errors.head())

In [ ]:
rf_pred_series = pd.Series(rf_pred, index=y_test.index)

In [ ]:
print(df_electricity.loc[8856, "local_timestamp"])
print(y_test.loc[8856])
print(rf_pred_series.loc[8856])
print(rf_error.loc[8856])

In [ ]:
error_index = 8856

surrounding_hours = df_electricity.loc[
    error_index - 6:error_index + 6,
    ["local_timestamp", "power_kw"]
]

print(surrounding_hours)

In [ ]:
print("Linear Regression MAE:", mae)
print("Linear Regression R²:", r2_score(y_test, y_pred))

print("Random Forest MAE:", rf_mae)
print("Random Forest R²:", rf_r2)

In [ ]:
similar_hours = df_electricity.loc[X_train.index]

similar_hours = similar_hours[
    (similar_hours["hour"] == 7) &
    (similar_hours["weekend"] == True) &
    (similar_hours["month"] == 12)
]

print("Number of similar observations:", len(similar_hours))

print("Average power:", similar_hours["power_kw"].mean())

print("Maximum power:", similar_hours["power_kw"].max())

print("Actual power at observation 8856:", y_test.loc[8856])

print("Random Forest prediction:", rf_pred_series.loc[8856])

In [ ]:
df_lag = df_electricity.sort_values("timestamp").copy()

df_lag["lag_1h"] = df_lag["power_kw"].shift(1)
df_lag["lag_24h"] = df_lag["power_kw"].shift(24)

valid_previous_hour = (
    df_lag["timestamp"].diff() == pd.Timedelta(hours=1)
)

df_lag.loc[~valid_previous_hour, "lag_1h"] = float("nan")

print(
    df_lag.loc[
        [8855, 8856, 8857],
        ["local_timestamp", "power_kw", "lag_1h"]
    ]
)

print(
    "Missing previous-hour values:",
    df_lag["lag_1h"].isna().sum()
)

## 21. Preparing to Train Your Improved Model

In [ ]:
df_lag = df_lag.dropna(subset=["lag_1h", "lag_24h"])

In [ ]:
df_lag["day_of_week"] = df_lag["local_timestamp"].dt.dayofweek

X = df_lag[["hour","month","day_of_week","lag_1h","lag_24h"]]

y = df_lag["power_kw"]

## 22. Splitting data

In [ ]:
split_index = int(len(df_lag) * 0.8)

In [ ]:
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

In [ ]:
rf_w_lag_model = RandomForestRegressor(random_state=42) 

rf_w_lag_model.fit(X=X_train, y=y_train)

## 24. Predict with lag

In [ ]:
rf_w_lag_pred = rf_w_lag_model.predict(X_test)

print(rf_w_lag_pred[:5])
print(y_test.head())

In [ ]:
mae_w_lag = mean_absolute_error(y_test,rf_w_lag_pred )
r2_score_w_lag = r2_score(y_test,rf_w_lag_pred )

print(mae_w_lag)
print(r2_score_w_lag)

In [ ]:

X_train_original = X_train[["hour","month","day_of_week"]]

X_test_original = X_test[["hour","month","day_of_week"]]

In [ ]:
original_ft_rf = RandomForestRegressor(random_state=42)

original_ft_rf.fit(X_train_original,y_train)

In [ ]:
original_ft_rf_pred = original_ft_rf.predict(X_test_original)

original_mae_w_lag = mean_absolute_error(y_test,original_ft_rf_pred )
original_r2_score_w_lag = r2_score(y_test,original_ft_rf_pred)

print(original_mae_w_lag)
print(original_r2_score_w_lag)

In [ ]:
original_rf_w_lag_error = abs(y_test - original_ft_rf_pred)

In [ ]:
rf_w_lag_error = abs(y_test - rf_w_lag_pred)

In [ ]:
rf_pred_w_lag_series = pd.Series(rf_w_lag_pred, index=y_test.index)

print(df_electricity.loc[8856, "local_timestamp"])
print(y_test.loc[8856])
print(rf_pred_w_lag_series.loc[8856])
print(rf_w_lag_error.loc[8856])

In [ ]:

print(original_rf_w_lag_error.head())
print(rf_w_lag_error.head())

print(original_rf_w_lag_error.shape)
print(rf_w_lag_error.shape)

abs_error_series = pd.Series([rf_w_lag_error,original_rf_w_lag_error])
print(abs_error_series)


In [ ]:
test_timestamps = df_electricity.loc[y_test.index, "local_timestamp"]

plt.figure(figsize=(12, 5))

plt.plot(
    test_timestamps.iloc[:168],
    y_test.iloc[:168],
    label="Actual Power"
)

plt.plot(
    test_timestamps.iloc[:168],
    rf_pred_w_lag_series.iloc[:168],
    label="Predicted Power w/ Lag"
)

plt.xlabel("Time")
plt.ylabel("Power (kW)")
plt.title("Random Forest: Actual vs Predicted Power w/Lag - First Test Week")
plt.legend()
plt.show()

In [ ]:
baseline_mae = mean_absolute_error(y_test, X_test["lag_1h"])
print("Previous-hour baseline MAE:", baseline_mae)
print("Random forest MAE:", mae_w_lag)

In [ ]:
next_hour_features = pd.DataFrame([{
    "hour": 14,         
    "month": 10,
    "day_of_week": 1,
    "lag_1h": 52.0,
    "lag_24h": 49.5
}])

next_hour_features

In [ ]:
predicted_power = rf_w_lag_model.predict(next_hour_features)

print(f"Predicted demand: {predicted_power[0]:.2f} kW")

In [ ]:
# Export before importing the predictor, including on a fresh checkout.
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_w_lag_model, MODEL_PATH)

import runpy

predictor_module = runpy.run_path(str(PROJECT_ROOT / "src" / "predictor.py"))
predict_power = predictor_module["predict_power"]
predict_power.__globals__["rf_w_lag_model"] = rf_w_lag_model


In [ ]:
print(predict_power(
    hour=14,
    month=10,
    day_of_week=1,
    lag_1h=52.0,
    lag_24h=49.5
))

In [ ]:
try:
    predict_power(30, 10, 1, 52.0, 49.5)
except ValueError as error:
    print("Expected validation error:", error)
else:
    raise AssertionError("Expected ValueError")


In [ ]:
predict_power(0, 10, 1, 52.0, 49.5)

In [ ]:
try:
    predict_power(14, 10, 1, -5.0, 49.5)
except ValueError as error:
    print("Expected validation error:", error)
else:
    raise AssertionError("Expected ValueError")


In [ ]:
prediction = predict_power(
    hour=14,
    month=10,
    day_of_week=1,
    lag_1h=52.0,
    lag_24h=49.5
)

print(prediction)

print(type(prediction))

In [ ]:
predict_power(14, 10, 2, 45.0, 49.5)

In [ ]:
prediction_inputs = {
    "hour": 14,
    "month": 10,
    "day_of_week": 2,
    "lag_1h": 45.0,
    "lag_24h": 49.5
}

In [ ]:
prediction = predict_power(**prediction_inputs)
#hour, month, day_of_week, lag_1h, lag_24h

print(prediction)

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
joblib.dump(rf_w_lag_model, MODEL_PATH)

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
assert hasattr(loaded_model, "predict")

In [ ]:
test_inputs = pd.DataFrame([{
    "hour": 14,
    "month": 10,
    "day_of_week": 2,
    "lag_1h": 45.0,
    "lag_24h": 49.5
}])

original_model_pred = rf_w_lag_model.predict(test_inputs)

loaded_model_pred = loaded_model.predict(test_inputs)

assert np.allclose(original_model_pred, loaded_model_pred)